# 04 — Embedding-based tabular neural network

**Owners:** Mirdula / Hashvitha  
**Role:** deep-learning benchmark for mixed numerical and high-cardinality data

Interview explanation: *“The network learns compact embeddings for categorical
identities such as cards, emails, and devices, combines them with standardized
numerical fraud signals, and optimizes a class-weighted binary objective.”*


## Feature engineering for this approach

- Numeric values: training median, missing indicator, standardization.
- Categories and identifier codes: `MISSING`, `OTHER`, and `UNKNOWN` tokens.
- Categories seen fewer than 20 times become the `OTHER` embedding during training.
- Future unseen values receive `UNKNOWN`, not an accidental training category.
- One embedding table is learned per categorical feature; no huge one-hot matrix.
- `BCEWithLogitsLoss(pos_weight=...)` addresses class imbalance.


In [ ]:
from pathlib import Path
_install_root = Path.cwd().resolve()
for _candidate in [_install_root, *_install_root.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)


In [ ]:
required = [
    PROCESSED_DIR / "train.parquet",
    PROCESSED_DIR / "validation.parquet",
    PROCESSED_DIR / "test.parquet",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run 00_shared_data_preparation.ipynb first. Missing: " + ", ".join(missing)
    )

train = pd.read_parquet(required[0])
validation = pd.read_parquet(required[1])
test = pd.read_parquet(required[2])

def stratified_debug_sample(frame, rows):
    if rows is None or rows >= len(frame):
        return frame
    return (
        frame.groupby("isFraud", group_keys=False)
        .apply(lambda group: group.sample(
            n=max(1, round(rows * len(group) / len(frame))),
            random_state=RANDOM_SEED,
        ), include_groups=True)
        .sort_values(["TransactionDT", "TransactionID"])
        .reset_index(drop=True)
    )

FAST_RUN = False  # Set True only to verify the notebook; never report these metrics.
if FAST_RUN:
    train = stratified_debug_sample(train, 60_000)
    validation = stratified_debug_sample(validation, 20_000)
    test = stratified_debug_sample(test, 20_000)

TARGET = "isFraud"
DROP_FROM_MODEL = ["isFraud", "TransactionID"]
X_train, y_train = train.drop(columns=DROP_FROM_MODEL), train[TARGET].astype("int8")
X_validation, y_validation = validation.drop(columns=DROP_FROM_MODEL), validation[TARGET].astype("int8")
X_test, y_test = test.drop(columns=DROP_FROM_MODEL), test[TARGET].astype("int8")

print("Train:", X_train.shape, "fraud rate:", f"{y_train.mean():.4%}")
print("Validation:", X_validation.shape, "fraud rate:", f"{y_validation.mean():.4%}")
print("Test:", X_test.shape, "fraud rate:", f"{y_test.mean():.4%}")


In [ ]:
MODEL_KEY = "neural_network"


In [ ]:
from datetime import datetime, timezone
from src.fraud_pipeline.artifacts import build_manifest, package_versions, write_json
from src.fraud_pipeline.evaluation import evaluate_binary_classifier, select_operating_threshold

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = ARTIFACT_ROOT / MODEL_KEY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("This run will be saved to:", RUN_DIR)


## Fit preprocessing and create tensors

The fitted preprocessor is part of the deployment bundle. Array creation uses CPU
RAM; switch to a Lightning machine with more memory if the kernel is killed.


In [ ]:
import joblib, torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score
from src.fraud_pipeline.neural import FraudTabularNetwork, embedding_dimension, network_from_config
from src.fraud_pipeline.preprocessing import NeuralTabularPreprocessor

preprocessor = NeuralTabularPreprocessor(rare_min_count=20).fit(X_train)
train_numeric, train_categorical = preprocessor.transform(X_train)
validation_numeric, validation_categorical = preprocessor.transform(X_validation)
print("Numerical tensor:", train_numeric.shape)
print("Categorical tensor:", train_categorical.shape)
print("Embedding fields:", len(preprocessor.cardinalities))


## Define the architecture

Embedding dimensions grow with vocabulary size but are capped at 32. Dense layers
use ReLU and dropout to learn nonlinear interactions while controlling overfitting.
Numeric inputs are already standardized, so batch-dependent normalization is not required.


In [ ]:
embedding_dimensions = [embedding_dimension(c) for c in preprocessor.cardinalities]
model_config = {
    "numeric_size": preprocessor.numeric_output_size,
    "cardinalities": preprocessor.cardinalities,
    "embedding_dimensions": embedding_dimensions,
    "categorical_columns": preprocessor.categorical_columns,
    "hidden_layers": [256, 128, 64],
    "dropout": [0.30, 0.20, 0.10],
}
model = FraudTabularNetwork(**{
    key: model_config[key]
    for key in ["numeric_size", "cardinalities", "embedding_dimensions"]
})
print(model)


## Train with mixed precision and validation PR-AUC early stopping

T4 is recommended. The best state is retained, not simply the final epoch.


In [ ]:
BATCH_SIZE = 4096
MAX_EPOCHS = 20
PATIENCE = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_dataset = TensorDataset(
    torch.from_numpy(train_numeric),
    torch.from_numpy(train_categorical),
    torch.from_numpy(y_train.to_numpy(dtype=np.float32)),
)
validation_dataset = TensorDataset(
    torch.from_numpy(validation_numeric),
    torch.from_numpy(validation_categorical),
    torch.from_numpy(y_validation.to_numpy(dtype=np.float32)),
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=device.type == "cuda")
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=device.type == "cuda")

negative, positive = np.bincount(y_train)
loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(negative / positive, device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

def predict_loader(model, loader):
    model.eval()
    probabilities, labels = [], []
    with torch.inference_mode():
        for numeric, categorical, target in loader:
            numeric, categorical = numeric.to(device), categorical.to(device)
            logits = model(numeric, categorical)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(target.numpy())
    return np.concatenate(probabilities), np.concatenate(labels)

best_pr_auc = -np.inf
best_state = None
epochs_without_improvement = 0
history = []
started = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for numeric, categorical, target in train_loader:
        numeric = numeric.to(device, non_blocking=True)
        categorical = categorical.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(numeric, categorical)
            loss = loss_function(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(target)

    validation_probability, validation_label = predict_loader(model, validation_loader)
    validation_pr_auc = average_precision_score(validation_label, validation_probability)
    epoch_loss = total_loss / len(train_dataset)
    history.append({"epoch": epoch, "train_loss": epoch_loss, "validation_pr_auc": validation_pr_auc})
    print(f"epoch={epoch:02d} loss={epoch_loss:.5f} validation_pr_auc={validation_pr_auc:.5f}")

    if validation_pr_auc > best_pr_auc + 1e-5:
        best_pr_auc = validation_pr_auc
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping")
            break

training_seconds = time.perf_counter() - started
model.load_state_dict(best_state)
model.to(device)
pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)


## Validation threshold and final holdout evaluation


In [ ]:
validation_probability, _ = predict_loader(model, validation_loader)
threshold_record = select_operating_threshold(y_validation, validation_probability, minimum_precision=0.10)
threshold = float(threshold_record["threshold"])
validation_metrics = evaluate_binary_classifier(y_validation, validation_probability, threshold)

test_numeric, test_categorical = preprocessor.transform(X_test)
test_dataset = TensorDataset(
    torch.from_numpy(test_numeric), torch.from_numpy(test_categorical),
    torch.from_numpy(y_test.to_numpy(dtype=np.float32)),
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=device.type == "cuda")
started = time.perf_counter()
test_probability, _ = predict_loader(model, test_loader)
prediction_seconds = time.perf_counter() - started
test_metrics = evaluate_binary_classifier(y_test, test_probability, threshold)
display(pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])[["pr_auc", "roc_auc", "precision", "recall", "f1", "brier_score"]])


## Save state dictionary, architecture, preprocessing, and predictions

Saving `state_dict` plus an explicit architecture config is safer and more portable
than pickling the entire Python model object.


In [ ]:
model_path = RUN_DIR / "model.pt"
torch.save({
    "model_state_dict": {key: value.cpu() for key, value in model.state_dict().items()},
    "model_config": model_config,
    "best_validation_pr_auc": best_pr_auc,
}, model_path)
preprocessor_path = RUN_DIR / "numeric_and_categorical_preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path, compress=3)
write_json(RUN_DIR / "model_config.json", model_config)
write_json(RUN_DIR / "category_vocabulary_summary.json", {
    column: {"cardinality": len(preprocessor.vocabularies[column]), "reserved": {"MISSING": 0, "UNKNOWN": 1, "OTHER": 2}}
    for column in preprocessor.categorical_columns
})
pd.DataFrame({"TransactionID": validation["TransactionID"], "isFraud": y_validation, "probability": validation_probability}).to_parquet(RUN_DIR / "validation_predictions.parquet", index=False)
pd.DataFrame({"TransactionID": test["TransactionID"], "isFraud": y_test, "probability": test_probability}).to_parquet(RUN_DIR / "test_predictions.parquet", index=False)
write_json(RUN_DIR / "threshold.json", threshold_record)
write_json(RUN_DIR / "metrics.json", {"validation": validation_metrics, "test": test_metrics})
write_json(RUN_DIR / "feature_schema.json", {"model": MODEL_KEY, "groups": preprocessor.groups, "categorical_columns": preprocessor.categorical_columns})
write_json(RUN_DIR / "training_config.json", {
    "model": MODEL_KEY, "run_id": RUN_ID, "random_seed": RANDOM_SEED,
    "fast_run": FAST_RUN, "training_seconds": training_seconds,
    "test_prediction_seconds": prediction_seconds, "device": str(device),
    "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS,
    "best_validation_pr_auc": best_pr_auc,
    "versions": package_versions(["numpy", "pandas", "scikit-learn", "torch", "joblib"]),
})


## Mandatory reload test


In [ ]:
loaded_preprocessor = joblib.load(preprocessor_path)
checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)
loaded_model = network_from_config(checkpoint["model_config"])
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()
sample_numeric, sample_categorical = loaded_preprocessor.transform(X_validation.iloc[:5])
with torch.inference_mode():
    after = torch.sigmoid(loaded_model(torch.from_numpy(sample_numeric), torch.from_numpy(sample_categorical))).numpy()
before = validation_probability[:5]
np.testing.assert_allclose(before, after, rtol=1e-5, atol=1e-7)
write_json(RUN_DIR / "manifest.json", build_manifest(RUN_DIR))
print("Reload test passed:", after)
print("Artifact directory:", RUN_DIR)


In [ ]:
# Optional promotion step: upload this versioned run to a private Cloudflare R2 bucket.
# Create these as Lightning secrets/environment variables; never paste keys into a cell.
UPLOAD_TO_R2 = False

if UPLOAD_TO_R2:
    import boto3
    required_names = [
        "R2_ENDPOINT_URL", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET_NAME"
    ]
    absent = [name for name in required_names if not os.getenv(name)]
    if absent:
        raise RuntimeError("Missing Lightning secrets: " + ", ".join(absent))
    client = boto3.client(
        "s3",
        endpoint_url=os.environ["R2_ENDPOINT_URL"],
        aws_access_key_id=os.environ["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )
    prefix = f"{MODEL_KEY}/{RUN_ID}"
    for local_path in RUN_DIR.rglob("*"):
        if local_path.is_file():
            key = f"{prefix}/{local_path.relative_to(RUN_DIR).as_posix()}"
            client.upload_file(str(local_path), os.environ["R2_BUCKET_NAME"], key)
    print(f"Uploaded to r2://{os.environ['R2_BUCKET_NAME']}/{prefix}/")
else:
    print("R2 upload skipped. Set UPLOAD_TO_R2=True after configuring Lightning secrets.")


## Interview checklist

Be ready to explain embeddings, reserved category tokens, numeric scaling, missing
indicators, class-weighted BCE loss, logits versus probabilities, dropout,
standardized inputs, mixed precision, dropout, and early stopping on validation PR-AUC.
